# Task:
- Load the two datasets
- Clean them (thoroughly)
- Measure at least 1 data engineering metric in the data cleaning process (ie dropped rows)
- Output the cleaned files as a local csv

# Stretch:
- Output the cleaned files into the local SSMS (use SQLAlchemy); create a new dB for this
- Refactor your code into modular functions


In [248]:
import pandas as pd
Transactions = pd.read_csv("Input Files/03_Library Systembook.csv")
Customers = pd.read_csv("Input Files/03_Library SystemCustomers.csv")
Error_Transactions =  Transactions.iloc[0:0].copy()
Error_Customers = Customers.iloc[0:0].copy()


In [249]:
fully_null = Customers[Customers.isnull().all(axis=1)].copy()
fully_null["Error_Desc"] = "Fully null entry"
Error_Customers = pd.concat([Error_Customers, fully_null])
Customers = Customers.drop(fully_null.index)

In [250]:
id_null = Customers[Customers["Customer ID"].isnull()].copy()
id_null["Error_Desc"] = "Null Customer ID"
Error_Customers = pd.concat([Error_Customers, id_null])
Customers = Customers.drop(id_null.index)

In [251]:
name_null = Customers[Customers["Customer Name"].isnull()].copy()
name_null["Error_Desc"] = "Null Customer ID"
Error_Customers = pd.concat([Error_Customers, name_null])
Customers["Customer Name"] = Customers["Customer Name"].fillna("Name Unknown")

In [252]:
fully_nullT = Transactions[Transactions.isnull().all(axis=1)].copy()
fully_nullT["Error_Desc"] = "Fully null entry"
Error_Transactions = pd.concat([Error_Transactions, fully_nullT])
Transactions = Transactions.drop(fully_nullT.index)

In [253]:
Transactions["Book checkout"] = Transactions["Book checkout"].str.strip('"')
Transactions["Book checkout"] = pd.to_datetime(Transactions["Book checkout"], format="%d/%m/%Y", errors="coerce")
Transactions["Book Returned"] = pd.to_datetime(Transactions["Book Returned"], format="%d/%m/%Y", errors="coerce")
invalid_dates = Transactions[Transactions["Book checkout"].isnull()].copy()
invalid_dates["Error_Desc"] = "Invalid Date"
Error_Transactions = pd.concat([Error_Transactions, invalid_dates])
Transactions = Transactions.drop(invalid_dates.index)

In [254]:
Transactions["Book Returned"] = pd.to_datetime(Transactions["Book Returned"], format="%d/%m/%Y", errors="coerce")

In [255]:
Transactions[["Borrow_Value", "Borrow_Interval"]] = Transactions["Days allowed to borrow"].str.split(" ", n=1, expand=True)

In [256]:
interval_map = {"week": 7, "weeks": 7, "day": 1, "days": 1, "month": 30, "months": 30, "year": 365, "years": 365}

Transactions["Borrow_Mult"] = Transactions["Borrow_Interval"].str.lower().map(interval_map)
Transactions["Borrow Limit (Days)"] = Transactions["Borrow_Value"].astype(int) * Transactions["Borrow_Mult"]
Transactions["Borrow Duration (Days)"] = (Transactions["Book Returned"] - Transactions["Book checkout"]).dt.days
Transactions = Transactions.drop(columns=["Borrow_Value", "Borrow_Mult", "Borrow_Interval", "Days allowed to borrow"])



In [257]:
invalid_customers = Transactions[~Transactions["Customer ID"].isin(Customers["Customer ID"])].copy()
invalid_customers["Error_Desc"] = "Invalid Customer ID"
Error_Transactions = pd.concat([Error_Transactions, invalid_customers])
Transactions = Transactions.drop(invalid_customers.index)

In [258]:
Transactions.to_csv("Output Files/Transactions_Clean.csv", index=False)
Customers.to_csv("Output Files/Customers_Clean.csv", index=False)
Error_Transactions.to_csv("Output Files/Transaction_Errors.csv", index=False)
Error_Customers.to_csv("Output Files/Customer_Errors.csv", index=False)

with open("Output Files/Error_Summary.txt", "w") as f:
    f.write("Customer Errors:\n")
    f.write(Error_Customers.groupby("Error_Desc").size().to_string())
    f.write("\n\nTransaction Errors:\n")
    f.write(Error_Transactions.groupby("Error_Desc").size().to_string())

